# Lab 0: Environment Check

Launch Jupyter from the `agentic-AI4-forensics` repo root after completing the setup steps in `01_instructions.md`.

This notebook will also try to locate the repo root automatically if your current working directory is inside the repo.

Before running the checks, make sure you copied `.env.example` to `.env`.

This notebook checks:
- Python and notebook paths
- required Python packages
- `.env` configuration
- Ollama endpoint connectivity
- Graphviz rendering

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.env.example').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError(
        'Could not find the repo root. Start Jupyter from the agentic-AI4-forensics folder, '
        'or open this notebook from inside that repo.'
    )

cwd = Path.cwd().resolve()
repo_root = find_repo_root(cwd)
env_path = repo_root / '.env'
src_path = repo_root / 'src'

print('Python executable:', sys.executable)
print('Current working directory:', cwd)
print('Detected repo root:', repo_root)
print('Repo .env exists:', env_path.exists())
print('Repo src exists:', src_path.exists())

if not env_path.exists():
    raise FileNotFoundError('Expected .env at the repo root. Copy .env.example to .env first.')

if not src_path.exists():
    raise FileNotFoundError('Expected src/ at the repo root.')

In [ ]:
import requests
from dotenv import dotenv_values
from graphviz import Digraph
from openai import OpenAI

print('Required packages imported successfully.')

In [ ]:
config = dotenv_values(env_path)
model = config.get('MODEL')
ollama_base_url = config.get('OLLAMA_BASE_URL')

print('MODEL:', model)
print('OLLAMA_BASE_URL:', ollama_base_url)

if not model:
    raise ValueError('MODEL is missing from .env')

if not ollama_base_url:
    raise ValueError('OLLAMA_BASE_URL is missing from .env')

In [ ]:
api_root = ollama_base_url.rstrip('/')
if api_root.endswith('/v1'):
    health_url = api_root[:-3] + '/api/tags'
else:
    health_url = api_root + '/api/tags'

try:
    response = requests.get(health_url, timeout=10)
    response.raise_for_status()
except requests.RequestException as exc:
    raise RuntimeError(
        f'Could not reach the configured Ollama endpoint at {health_url}. '
        'Check OLLAMA_BASE_URL in .env and confirm the server is reachable.'
    ) from exc

tags = response.json().get('models', [])

available_models = [item.get('name') for item in tags if item.get('name')]
print('Configured Ollama endpoint is reachable.')
print('Health check URL:', health_url)
print('Available models:', available_models)

if model not in available_models:
    model_family = model.split(':', 1)[0]
    close_matches = [name for name in available_models if model_family in name][:10]
    raise ValueError(
        f'Model {model!r} is not available at the configured Ollama endpoint. '
        f'Close matches: {close_matches or "none"}. '
        'Update .env or ask your instructor for the correct model.'
    )

In [ ]:
client = OpenAI(base_url=ollama_base_url, api_key='ollama')
response = client.chat.completions.create(
    model=model,
    messages=[
        {'role': 'system', 'content': 'You are a concise setup-check assistant.'},
        {'role': 'user', 'content': 'Reply with exactly: setup ok'}
    ]
)

print(response.choices[0].message.content)

In [ ]:
diagram = Digraph(comment='Setup Check')
diagram.attr(rankdir='LR')
diagram.node('env', '.env')
diagram.node('ollama', 'Configured Ollama Endpoint')
diagram.node('model', f'Model: {model}')
diagram.node('nb', 'Notebook')
diagram.edges([('env', 'nb'), ('ollama', 'nb'), ('model', 'ollama')])
diagram

## Complete

If all cells run successfully, your environment is ready for Lab 1.